# Video Highlight Extractor


## 1. Installing Libraries

In [ ]:
!pip -q install faster-whisper pandas librosa transformers scikit-learn

## 2. Imports and Configuration

In [ ]:
import os, re, json, subprocess
import numpy as np, pandas as pd, torch
from pathlib import Path
import librosa
from faster_whisper import WhisperModel
from transformers import AutoFeatureExtractor, WavLMForXVector, pipeline

WORK = Path("/content/work")
WORK.mkdir(parents=True, exist_ok=True)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Running on:", DEVICE)

CFG = dict(
    whisper_model="small",
    qa_thr=0.5,
    agree_thr=0.55,
    disagree_thr=0.55,
    max_answer_gap=8.0,
    top_k=5,
    buffer_y=20
)

Running on: cuda


## 3. Uploading Video & Extracting Audio

In [ ]:
def video_duration(video):
    out = subprocess.check_output([
        "ffprobe", "-v", "error", "-show_entries", "format=duration",
        "-of", "default=nw=1:nk=1", video
    ])
    return float(out)

def extract_audio(video):
    wav = str(WORK / "audio.wav")
    subprocess.run([
        "ffmpeg", "-y", "-i", video, "-vn", "-ac", "1", "-ar", "16000", wav,
        "-loglevel", "error"
    ], check=True)
    return wav

def fmt(t):
    t = int(t)
    return f"{t//3600:02d}:{(t%3600)//60:02d}:{t%60:02d}"

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import shutil

source = "/content/drive/MyDrive/video.mp4"
destination = "/content/work/video.mp4"

# Copy the file
shutil.copy(source, destination)


'/content/work/video.mp4'

In [ ]:
URL = "https://youtu.be/ufopNY5PO1U?si=U3gE2YNJz8vtUWJ_A"
VIDEO ="/content/work/video.mp4"
print("Video length:", fmt(video_duration(VIDEO)))
WAV = extract_audio(VIDEO)
print("Audio saved to:", WAV)
audio, sr = librosa.load(WAV, sr=16000)

Video length: 00:24:33
Audio saved to: /content/work/audio.wav


## 4. Speech-to-Text Transcription (Whisper)

In [ ]:
!pip -q install nvidia-cublas-cu12 nvidia-cudnn-cu12

import ctypes, glob, os, site

for base in site.getsitepackages():
    for pattern in [
        "nvidia/cublas/lib/libcublasLt.so.12*",
        "nvidia/cublas/lib/libcublas.so.12*",
        "nvidia/cudnn/lib/libcudnn.so.9*"
    ]:
        for lib in sorted(glob.glob(os.path.join(base, pattern))):
            try:
                ctypes.CDLL(lib, mode=ctypes.RTLD_GLOBAL)
            except OSError as e:
                pass

In [ ]:
def transcribe(wav):
    asr = WhisperModel(CFG["whisper_model"], device=DEVICE,
                       compute_type="float16" if DEVICE == "cuda" else "int8")
    audio_data, _ = librosa.load(wav, sr=16000)
    segs, _ = asr.transcribe(audio_data, vad_filter=True, beam_size=5, language="en")
    return [dict(start=s.start, end=s.end, text=s.text.strip()) for s in segs if s.text.strip()]

utts = transcribe(WAV)
print(f"Transcribed {len(utts)} raw utterances")

Transcribed 595 raw utterances


## 5. Sentence Merging


In [ ]:
def merge_sentences(utts, max_gap=1.2, max_len=20.0):
    merged = []
    for u in utts:
        prev = merged[-1] if merged else None
        if (prev and not re.search(r"[.?!]['\")\]]*$", prev["text"])
                and (u["start"] - prev["end"] <= max_gap)
                and (u["end"] - prev["start"] <= max_len)):
            prev["end"] = u["end"]
            prev["text"] += " " + u["text"]
        else:
            merged.append(dict(u))
    return merged

utts = merge_sentences(utts)
print(f"Cleaned to {len(utts)} sentence-level utterances")

Cleaned to 214 sentence-level utterances


## 6. Global Speaker Diarization (Differentiating the 2 People)


In [ ]:
## 6. Global Speaker Diarization
from sklearn.cluster import KMeans

# Load WavLM voice fingerprint model
_fe = AutoFeatureExtractor.from_pretrained("microsoft/wavlm-base-plus-sv")
_sv = WavLMForXVector.from_pretrained("microsoft/wavlm-base-plus-sv").to(DEVICE).eval()

def extract_embedding(audio, sr, start, end):
    s_sample, e_sample = int(start * sr), int(end * sr)
    seg = audio[s_sample:e_sample]
    if len(seg) < sr * 0.4:
        return None

    # If the segment is long, pick the 3-second window with the strongest speech energy
    # (Avoids picking intro music or silent gaps)
    target_len = int(3.0 * sr)
    if len(seg) > target_len:
        hop = int(0.5 * sr)
        best_start = 0
        max_rms = -1.0
        for i in range(0, len(seg) - target_len, hop):
            window = seg[i : i + target_len]
            rms = np.sqrt(np.mean(window ** 2))
            if rms > max_rms:
                max_rms = rms
                best_start = i
        clip = seg[best_start : best_start + target_len]
    else:
        clip = seg

    inp = _fe(clip, sampling_rate=16000, return_tensors="pt", padding=True)
    with torch.no_grad():
        emb = _sv(**{k: v.to(DEVICE) for k, v in inp.items()}).embeddings
    return torch.nn.functional.normalize(emb, dim=-1)[0].cpu().numpy()

# 1. Extract embeddings for all utterances
embeddings, valid_indices = [], []
for idx, u in enumerate(utts):
    emb = extract_embedding(audio, sr, u["start"], u["end"])
    if emb is not None:
        embeddings.append(emb)
        valid_indices.append(idx)

# 2. Cluster into 2 speakers using KMeans (prevents outlier isolation)
clustering = KMeans(n_clusters=2, n_init=10, random_state=42)
labels = clustering.fit_predict(np.array(embeddings))

# 3. Assign labels back to utterances
for idx in range(len(utts)):
    utts[idx]["speaker"] = "UNKNOWN"

for idx, label in zip(valid_indices, labels):
    utts[idx]["speaker"] = f"SPEAKER_{label}"

# Fill any short unassigned utterance by adjacent context
for i in range(len(utts)):
    if utts[i]["speaker"] == "UNKNOWN":
        utts[i]["speaker"] = utts[i - 1]["speaker"] if i > 0 else "SPEAKER_0"

# Display first 12 entries to verify speaker alternation
pd.DataFrame([
    dict(speaker=u["speaker"], start=fmt(u["start"]), end=fmt(u["end"]), text=u["text"])
    for u in utts
]).head(12)

preprocessor_config.json:   0%|          | 0.00/215 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/58.6k [00:00<?, ?B/s]

pytorch_model.bin: reconstructing file:   0%|          |  0.00B /  405MB            

pytorch_model.bin: downloading bytes:           |  0.00B            

model.safetensors: reconstructing file:   0%|          |  0.00B /  404MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/266 [00:00<?, ?it/s]

/usr/local/lib/python3.13/dist-packages/torch/nn/functional.py:6829: UserWarning: Support for mismatched key_padding_mask and attn_mask is deprecated. Use same type for both instead.
  key_padding_mask = _canonical_mask(


,speaker,start,end,text
0,SPEAKER_1,00:00:04,00:00:07,Welcome to Hard Talk. I'm Stephen Sacker.
1,SPEAKER_1,00:00:07,00:00:13,"Anger is a powerful force in politics, and the..."
2,SPEAKER_1,00:00:13,00:00:20,"Donald Trump, Brexit and a host of populist mo..."
3,SPEAKER_1,00:00:20,00:00:24,Where does it come from? How best to respond?
4,SPEAKER_1,00:00:24,00:00:38,"Well, one much-discussed, provocative perspect..."
5,SPEAKER_1,00:00:38,00:00:42,Is his diagnosis liberating or dangerous?
6,SPEAKER_1,00:00:42,00:01:01,"Jordan Peterson, welcome to Hard Talk."
7,SPEAKER_0,00:01:01,00:01:02,Thanks for the invitation.
8,SPEAKER_1,00:01:02,00:01:15,"At the beginning of this year, you wrote this ..."
9,SPEAKER_1,00:01:16,00:01:21,You have struck some sort of a chord. Why do y...


## 7. Acoustic Pitch Analysis

In [ ]:
def rising_pitch(audio, sr, start, end):
    t0 = max(start, end - 0.8)
    seg = audio[int(t0 * sr): int(end * sr)]
    if len(seg) < sr * 0.3:
        return 0.0
    f0 = librosa.yin(seg, fmin=70, fmax=400, sr=sr, frame_length=1024)
    f0 = f0[np.isfinite(f0)]
    if len(f0) < 6:
        return 0.0
    h = len(f0) // 2
    r = float(np.log(np.median(f0[h:]) / np.median(f0[:h])))
    return float(min(1.0, max(0.0, r / 0.2)))

## 8. Question-Answer Detector


In [ ]:
WH = r"^(who|what|when|where|why|how|which|whose|do|does|did|is|are|was|were|can|could|would|will|should|have|has|had|isn't|aren't|don't|doesn't)\b"

def last_question(text):
    sents = re.split(r"(?<=[.?!])\s+", text.strip())
    q = sents[-1] if sents else ""
    if not q.endswith("?"):
        return None
    if re.search(r",\s*(right|okay|ok|you know|yeah|isn't it)\?$", q, re.I):
        return None
    if len(q.split()) < 4:
        return None
    return q

def text_question_score(text):
    t = text.lower().strip()
    s = 0.0
    if "?" in t: s += 0.6
    if re.match(WH, t): s += 0.25
    return s

def detect_qa(utts, audio, sr):
    out = []
    for i in range(len(utts) - 1):
        q, a = utts[i], utts[i + 1]

        # Must be different speakers
        if q["speaker"] == a["speaker"]:
            continue

        qtext = last_question(q["text"])
        if qtext is None:
            continue

        n_words = len(a["text"].split())
        gap = a["start"] - q["end"]
        if gap > CFG["max_answer_gap"] or n_words < 4 or a["text"].strip().endswith("?"):
            continue

        base = text_question_score(qtext)
        qs = min(1.0, base + 0.15 * rising_pitch(audio, sr, q["start"], q["end"]))
        if qs < CFG["qa_thr"]:
            continue

        score = 0.6 * qs + 0.2 * min(n_words / 20, 1.0) + 0.2 * (1 - gap / CFG["max_answer_gap"])
        out.append(dict(
            type="qa",
            x=q["end"],
            score=float(score),
            text=f'[{q["speaker"]}] Q: {qtext} -> [{a["speaker"]}] A: {a["text"]}'
        ))
    return out

qa = detect_qa(utts, audio, sr)
print(f"Found {len(qa)} valid cross-speaker Q&A moments")

Found 14 valid cross-speaker Q&A moments


## 9. Agreement and Disagreement Detectors (NLI)


In [ ]:
_clf = pipeline("zero-shot-classification", model="facebook/bart-large-mnli",
                device=0 if DEVICE == "cuda" else -1)

HYP = {
    "agree": "The reply expresses agreement.",
    "disagree": "The reply expresses disagreement.",
    "neutral": "The reply is neutral or off-topic."
}

AGREE_CUES = ["i agree", "exactly", "absolutely", "you're right", "you are right", "that's true",
              "that's right", "totally", "definitely", "for sure", "i think so too", "good point",
              "100%", "couldn't agree more", "same here", "that makes sense"]

DISAGREE_CUES = ["i disagree", "i don't agree", "not really", "i don't think so", "i don't think that",
                 "that's not true", "that's not right", "that's wrong", "i wouldn't say",
                 "not necessarily", "i'm not sure about that", "no, ", "come on", "actually,",
                 "however", "on the contrary", "i beg to differ"]

def has_cue(text, cues):
    t = text.lower()
    return 1.0 if any(c in t for c in cues) else 0.0

def detect_agree_disagree(utts):
    pairs = []
    for i in range(1, len(utts)):
        prev, cur = utts[i-1], utts[i]
        # Only compare if speakers alternate and gap is small
        if prev["speaker"] != cur["speaker"] and (cur["start"] - prev["end"] <= CFG["max_answer_gap"]):
            prompt = f'Speaker A said: "{prev["text"][-250:]}" Speaker B replied: "{cur["text"][:250]}"'
            pairs.append((i, prompt))

    if not pairs:
        return [], []

    indices, prompts = zip(*pairs)
    res = _clf(list(prompts), candidate_labels=list(HYP.values()), hypothesis_template="{}", batch_size=16)
    if isinstance(res, dict):
        res = [res]

    agrees, disagrees = [], []
    for idx, r in zip(indices, res):
        cur, prev = utts[idx], utts[idx-1]
        p = dict(zip(r["labels"], r["scores"]))
        a_score = 0.7 * p[HYP["agree"]] + 0.3 * has_cue(cur["text"], AGREE_CUES)
        d_score = 0.7 * p[HYP["disagree"]] + 0.3 * has_cue(cur["text"], DISAGREE_CUES)
        mid_time = (cur["start"] + cur["end"]) / 2
        txt = f'[{prev["speaker"]}] {prev["text"]} -> [{cur["speaker"]}] {cur["text"]}'

        if a_score >= CFG["agree_thr"] and a_score > d_score:
            agrees.append(dict(type="agree", x=mid_time, score=float(a_score), text=txt))
        elif d_score >= CFG["disagree_thr"] and d_score > a_score:
            disagrees.append(dict(type="disagree", x=mid_time, score=float(d_score), text=txt))

    return agrees, disagrees

agree, disagree = detect_agree_disagree(utts)
print(f"Found {len(agree)} agreement and {len(disagree)} disagreement moments")

config.json:   0%|          | 0.00/1.15k [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 1.63GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/515 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

Found 1 agreement and 53 disagreement moments


## 10. Ranking & Generating Moments Table

In [ ]:
def select(cands, y, top_k):
    chosen = []
    for c in sorted(cands, key=lambda c: -c["score"]):
        if all(abs(c["x"] - k["x"]) >= 2 * y for k in chosen):
            chosen.append(c)
        if len(chosen) == top_k:
            break
    return sorted(chosen, key=lambda c: c["x"])

y = float(np.clip(CFG["buffer_y"], 15, 30))
rows = []
for name, cands in [("Question-Answer", qa), ("Agreement", agree), ("Disagreement", disagree)]:
    for c in select(cands, y, CFG["top_k"]):
        rows.append(dict(
            type=name,
            moment_x=fmt(c["x"]),
            x_seconds=round(c["x"], 1),
            score=round(c["score"], 3),
            transcript=c["text"]
        ))

moments = pd.DataFrame(rows)
moments.to_csv(WORK / "moments.csv", index=False)
print("Final Extracted Moments:")
moments

Final Extracted Moments:


,type,moment_x,x_seconds,score,transcript
0,Question-Answer,00:01:21,82.0,0.910,[SPEAKER_1] Q: Why do you think that is? -> [S...
1,Question-Answer,00:08:08,489.0,0.987,[SPEAKER_1] Q: Would you agree with that? -> [...
2,Question-Answer,00:14:35,876.0,1.000,[SPEAKER_1] Q: why is society overly feminized...
3,Question-Answer,00:16:26,987.0,0.910,[SPEAKER_0] Q: Where the temperamental differe...
4,Question-Answer,00:18:40,1121.0,0.921,[SPEAKER_1] Q: Do you think it was helpful for...
5,Agreement,00:08:13,494.0,0.979,[SPEAKER_1] Would you agree with that? -> [SPE...
6,Disagreement,00:05:12,312.5,0.686,[SPEAKER_1] But with your focus on what you se...
7,Disagreement,00:11:27,688.0,0.987,[SPEAKER_1] You don't seem to give any real im...
8,Disagreement,00:15:59,960.0,0.962,"[SPEAKER_1] But you say that because of that, ..."
9,Disagreement,00:21:28,1289.0,0.989,"[SPEAKER_1] But you, and the way that the publ..."


## WEEK 3

## 1.Installing Dependencies

In [ ]:
!pip -q install scenedetect opencv-python

## 2.Scene Change Detection with pyscenedetect

In [ ]:
import warnings
import logging
from scenedetect import open_video, SceneManager
from scenedetect.detectors import ContentDetector

def detect_scenes(video_path, threshold=27.0, min_scene_len=15):
    # Suppress moviepy regex SyntaxWarnings and deprecation notices
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")

        # Lower PySceneDetect internal logger to suppress info logs
        logging.getLogger('pyscenedetect').setLevel(logging.ERROR)

        video = open_video(video_path)
        scene_manager = SceneManager()
        scene_manager.add_detector(ContentDetector(threshold=threshold, min_scene_len=min_scene_len))

        scene_manager.detect_scenes(video, show_progress=False)
        scene_list = scene_manager.get_scene_list()

        scenes = []
        for s_start, s_end in scene_list:
            # Use .get_seconds() safely via .seconds property to prevent DeprecationWarning
            start_sec = s_start.seconds if hasattr(s_start, "seconds") else s_start.get_seconds()
            end_sec = s_end.seconds if hasattr(s_end, "seconds") else s_end.get_seconds()

            scenes.append({
                "start": float(start_sec),
                "end": float(end_sec)
            })
    return scenes

# Run scene detection cleanly
scenes = detect_scenes(VIDEO)
print(f"Detected {len(scenes)} visual scene/camera transitions in the video.")

/usr/local/lib/python3.13/dist-packages/moviepy/video/io/ffmpeg_reader.py:294: SyntaxWarning: invalid escape sequence '\d'
  lines_video = [l for l in lines if ' Video: ' in l and re.search('\d+x\d+', l)]
/usr/local/lib/python3.13/dist-packages/moviepy/video/io/ffmpeg_reader.py:367: SyntaxWarning: invalid escape sequence '\d'
  rotation_lines = [l for l in lines if 'rotate          :' in l and re.search('\d+$', l)]
/usr/local/lib/python3.13/dist-packages/moviepy/video/io/ffmpeg_reader.py:370: SyntaxWarning: invalid escape sequence '\d'
  match = re.search('\d+$', rotation_line)
/usr/local/lib/python3.13/dist-packages/moviepy/config_defaults.py:47: SyntaxWarning: invalid escape sequence '\P'
  IMAGEMAGICK_BINARY = r"C:\Program Files\ImageMagick-6.8.8-Q16\magick.exe"


Detected 132 visual scene/camera transitions in the video.


## 3.Boundary Snapping

In [ ]:
def snap_to_scene_boundaries(moment_x, buffer_y, scenes, total_duration, min_dur=30.0, max_dur=60.0):
    """
    Snaps raw [x - y, x + y] boundaries to the closest visual camera cuts,
    guaranteeing the clip duration stays between 30 and 60 seconds.
    """
    ideal_start = max(0.0, moment_x - buffer_y)
    ideal_end = min(total_duration, moment_x + buffer_y)

    # Default to ideal boundaries
    snapped_start = ideal_start
    snapped_end = ideal_end

    # 1. Snap start: find the scene cut nearest to ideal_start
    if scenes:
        cut_starts = [s["start"] for s in scenes if s["start"] <= moment_x]
        if cut_starts:
            # Pick the cut closest to ideal_start
            best_start = min(cut_starts, key=lambda t: abs(t - ideal_start))
            if abs(best_start - ideal_start) <= 5.0:  # Allow up to 5s shift
                snapped_start = best_start

        # 2. Snap end: find the scene cut nearest to ideal_end
        cut_ends = [s["end"] for s in scenes if s["end"] >= moment_x]
        if cut_ends:
            best_end = min(cut_ends, key=lambda t: abs(t - ideal_end))
            if abs(best_end - ideal_end) <= 5.0:
                snapped_end = best_end


    duration = snapped_end - snapped_start
    if duration < min_dur:
        needed = min_dur - duration
        snapped_start = max(0.0, snapped_start - needed / 2)
        snapped_end = min(total_duration, snapped_end + needed / 2)
    elif duration > max_dur:
        excess = duration - max_dur
        snapped_start += excess / 2
        snapped_end -= excess / 2

    return round(snapped_start, 2), round(snapped_end, 2)

## 4.Visual Cues Analysis

In [ ]:
import cv2

# Load OpenCV's pre-trained face detector (built directly into cv2)
face_cascade = cv2.CascadeClassifier(cv2.data.haarcascades + 'haarcascade_frontalface_default.xml')

def analyze_visual_cues(video_path, start_sec, end_sec, sample_fps=1):
    """
    Samples frames across the moment window to check speaker visibility
    and presence of clear facial reactions using OpenCV.
    """
    cap = cv2.VideoCapture(video_path)
    video_fps = cap.get(cv2.CAP_PROP_FPS) or 25.0
    frame_interval = int(video_fps / sample_fps)

    start_frame = int(start_sec * video_fps)
    end_frame = int(end_sec * video_fps)

    cap.set(cv2.CAP_PROP_POS_FRAMES, start_frame)

    frames_checked = 0
    faces_detected = 0

    current_frame = start_frame
    while current_frame <= end_frame:
        ret, frame = cap.read()
        if not ret:
            break

        if (current_frame - start_frame) % frame_interval == 0:
            frames_checked += 1
            # Convert to grayscale for Haar cascade
            gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
            faces = face_cascade.detectMultiScale(gray, scaleFactor=1.2, minNeighbors=4, minSize=(40, 40))

            if len(faces) > 0:
                faces_detected += 1

        current_frame += 1

    cap.release()

    face_presence_score = (faces_detected / frames_checked) if frames_checked > 0 else 0.0
    return round(face_presence_score, 2)

## 5.Master Integration

In [ ]:
total_vid_dur = video_duration(VIDEO)

enriched_moments = []
for idx, row in moments.iterrows():
    m_x = row["x_seconds"]

    # 1. Snap boundaries to scene transitions (30s to 60s duration)
    clip_start, clip_end = snap_to_scene_boundaries(
        moment_x=m_x,
        buffer_y=CFG["buffer_y"],
        scenes=scenes,
        total_duration=total_vid_dur,
        min_dur=30.0,
        max_dur=60.0
    )

    # 2. Analyze visual cues within the snapped window
    visual_score = analyze_visual_cues(VIDEO, clip_start, clip_end)

    # 3. Compute multi-modal combined score
    # 70% audio/dialogue score + 30% visual presence score
    final_score = round(0.7 * row["score"] + 0.3 * visual_score, 3)

    enriched_moments.append({
        "type": row["type"],
        "moment_x": row["moment_x"],
        "x_seconds": row["x_seconds"],
        "clip_start": clip_start,
        "clip_end": clip_end,
        "clip_duration": round(clip_end - clip_start, 1),
        "nlp_score": row["score"],
        "visual_score": visual_score,
        "final_multimodal_score": final_score,
        "transcript": row["transcript"]
    })

df_week3 = pd.DataFrame(enriched_moments).sort_values("final_multimodal_score", ascending=False)
df_week3.to_csv(WORK / "moments_week3.csv", index=False)
df_week3.head()

,type,moment_x,x_seconds,clip_start,clip_end,clip_duration,nlp_score,visual_score,final_multimodal_score,transcript
8,Disagreement,00:15:59,960.0,941.68,982.44,40.8,0.962,0.98,0.967,"[SPEAKER_1] But you say that because of that, ..."
7,Disagreement,00:11:27,688.0,670.84,708.00,37.2,0.987,0.89,0.958,[SPEAKER_1] You don't seem to give any real im...
9,Disagreement,00:21:28,1289.0,1265.72,1307.48,41.8,0.989,0.67,0.893,"[SPEAKER_1] But you, and the way that the publ..."
5,Agreement,00:08:13,494.0,474.00,510.72,36.7,0.979,0.68,0.889,[SPEAKER_1] Would you agree with that? -> [SPE...
2,Question-Answer,00:14:35,876.0,857.44,896.00,38.6,1.000,0.56,0.868,[SPEAKER_1] Q: why is society overly feminized...
